# Unit 3, Exercise 3: multiple linear regression

**Data note:** the module notebook reads `cars.csv` but the file was not supplied with it. The code and the
numbers match the W3Schools multiple regression tutorial, so I used that tutorial's data file
(W3Schools, no date). It reproduces the notebook's own coefficients exactly (0.00755095 and 0.00780526),
which confirms it is the same data.

Everything under **My changes** is mine.

*The comments at the end of the module's own code lines are mine, added to explain what each line does. The module's original comments are unchanged.*

In [1]:
import pandas                                       # tables
from sklearn import linear_model                    # scikit-learn's linear models

df = pandas.read_csv("cars.csv")                    # load the cars data

X = df[['Weight', 'Volume']]                        # two inputs: weight and engine volume
y = df['CO2']                                       # the output: CO2 emissions

regr = linear_model.LinearRegression()              # create the model
regr.fit(X, y)                                      # learn one coefficient per input, plus an intercept

#predict the CO2 emission of a car where the weight is 2300kg, and the volume is 1300cm3:
predictedCO2 = regr.predict([[2300, 1300]])         # CO2 for a 2,300 kg car with a 1,300 cm3 engine

print(predictedCO2)                                 # show the prediction

[107.2087328]


/home/z331992/.local/lib/python3.10/site-packages/sklearn/utils/validation.py:2749: UserWarning: X does not have valid feature names, but LinearRegression was fitted with feature names
  warnings.warn(


# Coefficient

The coefficient is a factor that describes the relationship with an unknown variable.
In this case, we can ask for the coefficient value of weight against CO2, and for volume against CO2. The answer(s) we get tells us what would happen if we increase, or decrease, one of the independent values.

In [2]:
print(regr.coef_)                                   # CO2 change per extra kg, and per extra cm3

[0.00755095 0.00780526]


The result array represents the coefficient values of weight and volume.

Weight: 0.00755095
Volume: 0.00780526

These values tell us that if the weight increase by 1kg, the CO2 emission increases by 0.00755095g.

And if the engine size (Volume) increases by 1 cm3, the CO2 emission increases by 0.00780526 g.

I think that is a fair guess, but let test it!

We have already predicted that if a car with a 1300cm3 engine weighs 2300kg, the CO2 emission will be approximately 107g.

What if we increase the weight with 1000kg (from 2300 to 3300) what will be the CO2 emission?

Ans: 107.2087328 + (1000 * 0.00755095) = 114.75968

In [3]:
predictedCO2 = regr.predict([[3300, 1300]])         # same engine, 1,000 kg heavier

print(predictedCO2)                                 # show the prediction

[114.75968007]


/home/z331992/.local/lib/python3.10/site-packages/sklearn/utils/validation.py:2749: UserWarning: X does not have valid feature names, but LinearRegression was fitted with feature names
  warnings.warn(


## My changes

### 1. How good is this model?

In [4]:
from sklearn.metrics import r2_score                 # R² score (imported for reference; .score() below gives the same)
from sklearn.model_selection import cross_val_score, LeaveOneOut   # tools to test a model on data it hasn't seen
import numpy as np                                   # numpy for the averages

print(f"R² on the 36 cars it was fitted on: {regr.score(X, y):.3f}")   # how well it fits its own training data
# Leave-one-out: fit on 35 cars, predict the 36th, and repeat for every car in turn
loo = cross_val_score(linear_model.LinearRegression(), X, y, cv=LeaveOneOut(),  # fit on 35 cars, test on the 36th, 36 times
                      scoring="neg_mean_absolute_error")   # scikit-learn reports errors as negative numbers
print(f"Average error predicting a car it has NOT seen (leave-one-out): {-loo.mean():.1f} g CO2")   # flip the sign back
# The simplest possible "model": always guess the average CO2. Any real model should beat this.
print(f"Average error if you just guess the mean every time:             {np.mean(np.abs(y - y.mean())):.1f} g CO2")  # the do-nothing baseline any model should beat
print()                                              # blank line
print(df[["Weight", "Volume", "CO2"]].corr().round(2))   # correlations between the inputs and the output

R² on the 36 cars it was fitted on: 0.377
Average error predicting a car it has NOT seen (leave-one-out): 5.6 g CO2
Average error if you just guess the mean every time:             5.9 g CO2

        Weight  Volume   CO2
Weight    1.00    0.75  0.55
Volume    0.75    1.00  0.59
CO2       0.55    0.59  1.00


### 2. Which input matters more?

In [5]:
from sklearn.preprocessing import StandardScaler     # rescales each column to mean 0 and standard deviation 1

scaled = StandardScaler().fit_transform(X)           # weight and volume on the same scale
regr_scaled = linear_model.LinearRegression().fit(scaled, y)   # refit the model on the scaled inputs
print("Raw coefficients (per kg, per cm3):        ", np.round(regr.coef_, 5))       # the module's coefficients
print("Coefficients per 1 standard deviation:      ", np.round(regr_scaled.coef_, 2))   # now directly comparable
print(f"Weight spread: {X['Weight'].std():.0f} kg.  Volume spread: {X['Volume'].std():.0f} cm3.")   # why they differ

Raw coefficients (per kg, per cm3):         [0.00755 0.00781]
Coefficients per 1 standard deviation:       [1.8  2.99]
Weight spread: 242 kg.  Volume spread: 389 cm3.


### 3. Changing the inputs

In [6]:
import pandas as pd                                  # tables
cars = pd.DataFrame([[2300, 1300], [3300, 1300], [2300, 2300], [790, 1000], [5000, 6000]],  # five made-up cars, including the module's two examples
                    columns=["Weight", "Volume"])     # five made-up cars, including the module's two examples
cars["predicted CO2"] = regr.predict(cars[["Weight", "Volume"]]).round(1)   # the model's prediction for each
# Flag whether each car sits inside the range of the 36 cars the model learned from
cars["inside the data range?"] = ((cars.Weight.between(X.Weight.min(), X.Weight.max())) &  # True only if both weight and volume are inside the training range
                                  (cars.Volume.between(X.Volume.min(), X.Volume.max())))
print(f"Data range: weight {X.Weight.min()}-{X.Weight.max()} kg, volume {X.Volume.min()}-{X.Volume.max()} cm3")  # the range of the 36 training cars
print(cars.to_string(index=False))                   # print the table without the row numbers

Data range: weight 790-1746 kg, volume 900-2500 cm3
 Weight  Volume  predicted CO2  inside the data range?
   2300    1300          107.2                   False
   3300    1300          114.8                   False
   2300    2300          115.0                   False
    790    1000           93.5                    True
   5000    6000          164.3                   False


## What I took from it


- **This model is barely better than guessing.** R² is 0.377, so it explains about 38% of the variation in
  CO2 on the cars it was fitted to. On cars it hasn't seen, it is off by 5.6 g on average. Just guessing the
  average CO2 every time is off by 5.9 g. So weight and volume together buy us 0.3 g. I wouldn't have known
  that from the notebook, which only shows predictions and never checks them.
- **Weight and volume overlap.** They correlate at 0.75 with each other. Heavier cars have bigger engines,
  so the model can't cleanly separate their effects. This is called collinearity.
- **Raw coefficients can mislead.** Weight and volume look almost equal (0.0076 vs 0.0078). That's per kg and
  per cm³, which are different units. Put on the same scale (per standard deviation), volume matters about
  1.7 times more than weight.
- **The notebook's own example is outside the data.** It predicts a 2,300 kg car, but the heaviest car in the
  data is 1,746 kg. The 3,300 kg example is even further out. The numbers come out looking precise, which is
  the dangerous part.